In [ ]:
# 셀 1: 패키지 설치
!pip install -q pyngrok bitsandbytes transformers peft accelerate
print('완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.8 MB/s eta 0:00:00
완료


In [ ]:
# 기존 ngrok 프로세스 강제 종료
subprocess.run(['pkill', '-f', 'ngrok'], capture_output=True)
time.sleep(2)

# Colab Secret 사용
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

public_url = ngrok.connect(8000)
print(f"외부 URL: {public_url}")

외부 URL: NgrokTunnel: "https://lankiness-revered-skilled.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import json
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import uvicorn
import threading
from google.colab import drive

drive.mount('/content/drive')

# 모델 로드
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
CHECKPOINT = '/content/drive/MyDrive/remind_finetune/qwen_finetuned'
SYSTEM_PROMPT = "당신은 앱 사용자의 생활 로그와 PHQ-9 분석 데이터를 바탕으로 의사용 임상 요약 리포트를 생성하는 AI입니다. 반드시 주요 증상, 위험요인, 개선요인 3가지 섹션으로만 작성하고 진료 권고나 결론은 포함하지 마세요."

print("모델 로드 중...")

# torch.load 패치
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load
def patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return torch._original_load(*args, **kwargs)
torch.load = patched_torch_load


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='cuda:0',
    trust_remote_code=True,
)

model.load_adapter(CHECKPOINT)
model.eval()
print("모델 로드 완료!")

# FastAPI 앱
app = FastAPI()

class ReportRequest(BaseModel):
    input_data: dict

@app.post("/generate")
async def generate(req: ReportRequest):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(req.input_data, ensure_ascii=False)}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return {"output": result}

# 백그라운드에서 서버 실행
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print(f"Qwen 서버 실행 중!")
print(f"외부 URL: {public_url}/generate")

Mounted at /content/drive
모델 로드 중...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

모델 로드 완료!
Qwen 서버 실행 중!
외부 URL: NgrokTunnel: "https://lankiness-revered-skilled.ngrok-free.dev" -> "http://localhost:8000"/generate
